In [0]:
with origin_tb as (
  select 'webos23' as tb_nm, A.*
  from aic_data_ods.tlamp.normal_log_webos23 A
  where 1=1
    and date_ym between '2025-09' and '2026-03'
    and context_name="com.webos.app.homeconnect"
    and message_id in ("NL_HC_DEVICELIST", "NL_HC_CONTROL", "NL_HC_MODERUN", "NL_HC_MODELIST", "NL_HC_SELECT_TIPS")
  union all
  select 'webos24' as tb_nm, B.*
  from aic_data_ods.tlamp.normal_log_webos24 B
  where 1=1
    and date_ym between '2025-09' and '2026-03'
    and context_name="com.webos.app.homeconnect"
    and message_id in ("NL_HC_DEVICELIST", "NL_HC_CONTROL", "NL_HC_MODERUN", "NL_HC_MODELIST", "NL_HC_SELECT_TIPS")
  union all
  select 'webos25' as tb_nm, C.*
  from aic_data_ods.tlamp.normal_log_webos25 C
  where 1=1
    and date_ym between '2025-09' and '2026-03'
    and context_name="com.webos.app.homeconnect"
    and message_id in ("NL_HC_DEVICELIST", "NL_HC_CONTROL", "NL_HC_MODERUN", "NL_HC_MODELIST", "NL_HC_SELECT_TIPS")
),
filtered_tb as (
  select *
  from origin_tb
  where message_id in ("NL_HC_MODELIST", "NL_HC_DEVICELIST","NL_HC_SELECT_TIPS")
),
last_log as (
  select
    tb_nm, date_ym,mac_addr, X_User_Number, X_Device_Country, normal_log, context_name, message_id, log_create_time,
    row_number() over (
      partition by tb_nm, date_ym, message_id, mac_addr, X_User_Number
      order by log_create_time desc
    ) as rn
  from filtered_tb
)
select 
  -- *
  message_id, count(1)
  -- tb_nm, date_ym, message_id, count(1), count(distinct mac_addr) AS count
from (
  select tb_nm, date_ym,mac_addr, X_User_Number, X_Device_Country, normal_log, context_name, message_id
  from origin_tb
  where not message_id in ("NL_HC_MODELIST", "NL_HC_DEVICELIST","NL_HC_SELECT_TIPS")
  union all
  select tb_nm, date_ym,mac_addr, X_User_Number, X_Device_Country, normal_log, context_name, message_id
  from last_log
  where rn = 1
)
where 1=1
  and date_ym between '2025-09' and '2026-03'
  and context_name="com.webos.app.homeconnect"
  and message_id in ("NL_HC_DEVICELIST", "NL_HC_CONTROL", "NL_HC_MODERUN", "NL_HC_MODELIST", "NL_HC_SELECT_TIPS")
group by 
  -- tb_nm, date_ym, 
  message_id
-- order by 1,2,3,4